# 📽️ **Reprojection**

> ### 📌 **TL;DR**
>
> This Jupyter notebook has been created to compare different features with several open source Python libraries for rasters management.
>
> This notebook compares raster reprojection capabilities across several libraries.
>
> The following libraries will be considered :
>- `rasterio`
>- `rioxarray`
>- `odc-geo`
>- `geoutils`

In [ ]:
import numpy as np

import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show

import rioxarray

import odc.geo.xr
from odc.geo.xr import xr_reproject

import geoutils as gu

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import seaborn #cmap mako

In [ ]:
#path to the raster object
raster_path = "../data/rasters/105005005BDEA700-visual.tif"

## **Reprojection (without RPCs)**

Reprojecting a raster is available in every library considered in this study.

It consists of converting a raster to a different coordinate reference system (CRS).

It can be done for many different reasons:
- to make sure the image is correctly aligned and standardized with other data
- to work in a specific projection suitable for the intended use (for instance, distance or area measurements)
- orthorectifying data with RPCs/ GCPs
- to adapt the resolution and pixels size
- to change the grid
- to translate or apply a rotation
- to publish or share data on some platforms/ web services that require a specific projection
- ...


### • *rasterio*

In [ ]:
ds_rasterio = rasterio.open(raster_path)

In [ ]:
for key, value in ds_rasterio.meta.items():
    print(f"{key} : {value}")

In [ ]:
show(ds_rasterio.read(1), cmap="mako")

In [ ]:
print(f"Dataset's CRS : {ds_rasterio.crs}")

Our raster is currently projected in `EPSG:32627`, and we want it to be converted to `EPSG:4326`.

`rasterio`'s workflow is convoluted:

- We will use the [`calculate_default_transform()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.warp.html#rasterio.warp.calculate_default_transform) function to transform bounds to target coordinate system, calculate resolution, and returns destination transform and dimensions.
- We will then use [`reproject()`](https://rasterio.readthedocs.io/en/stable/api/rasterio.warp.html#rasterio.warp.reproject) to reproject our raster to the new CRS. The reprojection with `rasterio` is performed eagerly. Since it works on numpy arrays, the reading is done instantly, and reprojecting triggers calculations and load data into memory.

In `rasterio`, unlike other libraries, the number of arguments that need to be specified as inputs for the reprojection makes it non-trivial.
Moreover, the metadata management is not an easy task: we first need to copy the metadata of our input raster, and pass them as an argument during the reprojection, so the output raster has its own metadata, which is not really convenient.

In [ ]:
dst_crs = "EPSG:4326" #targeted CRS

dst_path = "../outputs/rasterio_reproj.tif"

with rasterio.open(raster_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds)
    kwargs = src.meta.copy() #copy metadata
    kwargs.update({
        'crs': dst_crs,
        'transform': transform,
        'width': width,
        'height': height
    })

    with rasterio.open(dst_path, 'w', **kwargs) as dst:
        for i in range(1, src.count + 1): #band-by-band reprojection
            reproject(
                source=rasterio.band(src, i),
                destination=rasterio.band(dst, i),
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=transform,
                dst_crs=dst_crs,
                resampling=Resampling.nearest)

We can then plot our new image and its associated metadata:

In [ ]:
with rasterio.open(dst_path) as rasterio_reproj:
    for key, value in rasterio_reproj.meta.items():
        print(f"{key} : {value}")
    show(rasterio_reproj.read(1), cmap="mako")

### • *rioxarray*

With rioxarray, we will use the [`.reproject()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.reproject) function:

In [ ]:
da_rxr = rioxarray.open_rasterio(raster_path, chunks={'x' : 2048, 'y' : 2048})

Here are the metadata of our dataset:

In [ ]:
print(f"""
Dimensions : {da_rxr.dims}
Number of band(s) : {da_rxr.rio.count}
Data type : {da_rxr.dtype}
Shape : {da_rxr.shape}
Size : {da_rxr.rio.height} rows x {da_rxr.rio.width} columns
CRS : {da_rxr.rio.crs}
Resolution : {da_rxr.rio.resolution()}
Bounds : {da_rxr.rio.bounds()}
Transform : {da_rxr.rio.transform()}
Attributes : {da_rxr.attrs}
""")

`rioxarray` greatly simplifies the metadata management compared to rasterio.

The [`.reproject()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.reproject) function is more straightforward than in `rasterio`, we just need to specify the targeted CRS.

Since we have opened our raster with chunks, data are lazy and stored as dask tasks. When calling the reproject function, it will create a `dask` graph but calculations are only executed when using `.compute()`, or when saving the array to a new file.

In [ ]:
dst_crs = "EPSG:4326" #targeted CRS

rxr_reproj = da_rxr.rio.reproject(dst_crs)

We can now verify the new CRS and image:

In [ ]:
print(f"""
Dimensions : {rxr_reproj.dims}
Number of band(s) : {rxr_reproj.rio.count}
Data type : {rxr_reproj.dtype}
Shape : {rxr_reproj.shape}
Size : {rxr_reproj.rio.height} rows x {rxr_reproj.rio.width} columns
CRS : {rxr_reproj.rio.crs}
Resolution : {rxr_reproj.rio.resolution()}
Bounds : {rxr_reproj.rio.bounds()}
Transform : {rxr_reproj.rio.transform()}
Attributes : {rxr_reproj.attrs}
""")

And we can plot the new image:

In [ ]:
rxr_reproj[0].plot.imshow(cmap="mako")

### • *odc-geo*

With `odc-geo`, we will use the [`.xr_reproject()`](https://odc-geo.readthedocs.io/en/latest/_api/odc.geo.xr.xr_reproject.html) function to reproject our raster

In [ ]:
ds_odcgeo = rioxarray.open_rasterio(raster_path, chunks={'x' : 2048, 'y' : 2048})

We can simply pass the image and the desired CRS as arguments.

`odc-geo` has implemented a lazy version of the reprojection, called `xr_reproject`. This function will create new dask tasks for reprojection but will not trigger instant calculations.

In [ ]:
dst_crs = "EPSG:4326"

odcgeo_reproj = xr_reproject(ds_odcgeo, dst_crs)

Our new raster has now the expected CRS:

In [ ]:
print(odcgeo_reproj.rio.crs)

Let's plot the newly created raster (the computation should be triggered by the plot and this may take a bit of time):

In [ ]:
odcgeo_reproj[0].plot.imshow(cmap="mako")

### • *geoutils*

In geoutils, we will use the [`.reproject()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.reproject.html#geoutils.Raster.reproject) function:

In [ ]:
ds_geoutils = gu.Raster(raster_path)

In [ ]:
print(f"Dataset's CRS : {ds_geoutils.crs}")

To reproject, we can simply pass the targeted CRS as an argument.

When using the `.reproject` function, data will be loaded instantly, will do the reprojection using memory and will return the newly created object with the reprojection result.

In [ ]:
gu_reprojected = ds_geoutils.reproject(crs=dst_crs)

However, `geoutils` will release soon an `xarray` accessor and with it a lazy version of its reprojection, leveraging `dask` as seen in the documentation:
 > reprojection can be computed out-of-memory in multiprocessing, by passing a MultiprocConfig object. The reprojected raster is written to disk under the path specified in the configuration.

In [ ]:
print(f"Reprojected Dataset's CRS : {gu_reprojected.crs}")

We can finally plot the new raster:

In [ ]:
gu_reprojected.plot(bands=1, cmap="mako")

## **Reproject match**

Reproject match is a feature only available in `rioxarray` and `geoutils`.

This feature is useful to reproject an image to match the exact projection, resolution and grid of another raster without extracting these information by hand and repassing them to the classic reprojection function, making it a very useful shortcut to simplify this task.

In this example we will reproject a DEM (Digital Elevation Model) on a satellite image with an different CRS.

### • *rioxarray*

With `rioxarray`, we can use the [`.reproject_match()`](https://corteva.github.io/rioxarray/stable/examples/reproject_match.html) function:

In this example we will reproject a DEM imagery of Sardinia with an incorrect CRS to align it with another raster that has the correct one.

In [ ]:
#path of the sat image
raster_path = "../data/rasters/T32TNK_20251113T101159_TCI.tif"

In [ ]:
#path of the dem with incorrect CRS
dem_incorrect_crs_path = "../data/rasters/sard1.tif"

In [ ]:
da_rxr_valid = rioxarray.open_rasterio(raster_path)

da_rxr_dem_invalid = rioxarray.open_rasterio(dem_incorrect_crs_path)

In [ ]:
print(f"""
VALID RASTER ATTRIBUTES\n
shape: {da_rxr_valid.rio.shape}
resolution: {da_rxr_valid.rio.resolution()}
bounds: {da_rxr_valid.rio.bounds()}
CRS: {da_rxr_valid.rio.crs}
""")

Here is the reference raster that we want to match:

In [ ]:
da_rxr_valid.plot.imshow()
plt.show()

In [ ]:
print(f"""
DEM INCORRECT CRS ATTRIBUTES\n
shape: {da_rxr_dem_invalid.rio.shape}
resolution: {da_rxr_dem_invalid.rio.resolution()}
bounds: {da_rxr_dem_invalid.rio.bounds()}
CRS: {da_rxr_dem_invalid.rio.crs} <-- to be changed
""")

We can plot the DEM with incorrect CRS, but first we will convert nodata values to NaNs:

In [ ]:
da_rxr_dem_invalid = da_rxr_dem_invalid.where(da_rxr_dem_invalid != da_rxr_dem_invalid.rio.nodata)

In [ ]:
da_rxr_dem_invalid.sel(band=1).plot.imshow(cmap="mako", vmin=0)

DEM CRS is `IGNF:ETRS89UTM32` and we want to align it onto the satellite image one: `EPSG:32632` (CRS of the reference raster)

We can use the [`reproject_match()`](https://corteva.github.io/rioxarray/stable/rioxarray.html#rioxarray.raster_array.RasterArray.reproject_match) function, we just have to pass the raster correctly projected as an argument:

In [ ]:
rxr_repr_match = da_rxr_dem_invalid.rio.reproject_match(da_rxr_valid)

If we check the newly projected raster, its CRS has been changed accordingly to our referenced raster:

In [ ]:
print(f"""
REPROJECTED RASTER ATTRIBUTES\n
shape: {rxr_repr_match.rio.shape}
resolution: {rxr_repr_match.rio.resolution()}
bounds: {rxr_repr_match.rio.bounds()}
CRS: {rxr_repr_match.rio.crs}
""")

We can now plot our reference raster with the reprojected DEM on top, as follows:

In [ ]:
fig,ax = plt.subplots(figsize=(10, 10))

da_rxr_valid.plot.imshow(ax=ax)

rxr_repr_match.where(rxr_repr_match!=rxr_repr_match.rio.nodata).sel(band=1).plot.imshow(ax=ax, cmap="mako", add_colorbar=True, vmin=0)

plt.show()

### • *geoutils*

To reproject match using `geoutils`, we will reuse the [`.reproject()`](https://geoutils.readthedocs.io/en/stable/gen_modules/geoutils.Raster.reproject.html) function already seen previously, but this time we will pass the argument `ref=` with another raster, so our image will be reprojected according to the resolution, bounds and CRS of this referenced raster.


In [ ]:
gu_rast_valid = gu.Raster(raster_path)

gu_rast_invalid = gu.Raster(dem_incorrect_crs_path)

In [ ]:
gu_rast_valid.info()

In [ ]:
gu_rast_invalid.info()

As previously, we can reuse the `reproject()` function, but this time we pass our valid raster as an argument:

In [ ]:
gu_reproj = gu_rast_invalid.reproject(gu_rast_valid)

The reprojected raster has now the right CRS :

In [ ]:
gu_reproj.info()

Finally, we can plot our reprojected DEM on top of our reference raster, as follows:

In [ ]:
gu_rast_valid.plot()
gu_reproj.plot(cmap="mako")